# Conferência do CO 1001

Este notebook faz 4 etapas:

1. Lê a matriz MSC corretamente (`skiprows=1`).
2. Filtra na matriz apenas `saldo final` do `CO 1001` (`TIPO4 = CO` e `IC4 = 1001`).
3. Constrói no arquivo do sistema as chaves equivalentes da MSC (`IC2 = funcao + sub_funcao` e `IC3 = ano_fonte + fonte`).
4. Aplica no extrato detalhado do sistema a mesma regra usada no relatório.
5. Compara os valores da matriz com o resultado filtrado do sistema.

Ajuste apenas os nomes dos arquivos e, se necessário, os aliases das colunas do extrato do sistema.

In [23]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.float_format = lambda x: f'{x:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')


def normalizar_texto(texto):
    texto = '' if texto is None else str(texto).strip()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
    for alvo in ['[', ']', '(', ')', '.', '-', '/', '\\']:
        texto = texto.replace(alvo, ' ')
    return ' '.join(texto.upper().split())


def localizar_colunas(df, aliases):
    mapa = {normalizar_texto(col): col for col in df.columns}
    encontradas = {}

    for destino, candidatos in aliases.items():
        for candidato in candidatos:
            chave = normalizar_texto(candidato)
            if chave in mapa:
                encontradas[destino] = mapa[chave]
                break

    faltantes = [col for col in aliases if col not in encontradas]
    if faltantes:
        raise KeyError(f'Colunas nao encontradas no extrato do sistema: {faltantes}')

    return df.rename(columns={origem: destino for destino, origem in encontradas.items()})


def converter_valor(serie):
    serie = serie.astype(str).str.strip()
    serie = serie.replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

    tem_virgula = serie.str.contains(',', na=False)
    tem_ponto = serie.str.contains('.', regex=False, na=False)

    serie = np.where(
        tem_virgula & tem_ponto,
        serie.str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        np.where(
            tem_virgula,
            serie.str.replace(',', '.', regex=False),
            serie,
        ),
    )

    return pd.to_numeric(pd.Series(serie), errors='coerce')


In [24]:
ano = '2026'
mes = '02'

arquivo_matriz = Path(f'msc_{mes}_{ano}.csv')
arquivo_sistema = Path('consulta_sistema_co1001.csv')

co_alvo = '1001'
tipo_co = 'CO'
tipo_valor_matriz = 'ending_balance'


In [25]:
# A matriz tem uma primeira linha com identificacao do arquivo, por isso usamos skiprows=1.
df_matriz = pd.read_csv(
    arquivo_matriz,
    sep=';',
    dtype=str,
    encoding='latin-1',
    skiprows=1,
)

df_matriz.columns = [col.strip() for col in df_matriz.columns]
for col in df_matriz.columns:
    df_matriz[col] = df_matriz[col].astype(str).str.strip()

df_matriz['VALOR_NUM'] = converter_valor(df_matriz['VALOR'])

matriz_co1001 = df_matriz.loc[
    (df_matriz['TIPO_VALOR'] == tipo_valor_matriz)
    & (df_matriz['TIPO4'] == tipo_co)
    & (df_matriz['IC4'] == co_alvo)
].copy()

# Chaves equivalentes para comparacao com o arquivo do sistema.
matriz_co1001['ic2_equivalente'] = matriz_co1001['IC2'].str.zfill(5)
matriz_co1001['ic3_equivalente'] = matriz_co1001['IC3'].str.zfill(4)
matriz_co1001['natureza_despesa_codigo'] = matriz_co1001['IC5'].str.zfill(8)
matriz_co1001['elemento_despesa_codigo'] = matriz_co1001['IC5'].str[-2:]

print('Linhas da matriz para CO 1001 (saldo final):', len(matriz_co1001))
print('Total da matriz para CO 1001 (saldo final):', matriz_co1001['VALOR_NUM'].sum())

matriz_co1001.head()


Linhas da matriz para CO 1001 (saldo final): 879
Total da matriz para CO 1001 (saldo final): 12839974512.91


,CONTA,IC1,TIPO1,IC2,TIPO2,IC3,TIPO3,IC4,TIPO4,IC5,...,IC6,TIPO6,VALOR,TIPO_VALOR,NATUREZA_VALOR,VALOR_NUM,ic2_equivalente,ic3_equivalente,natureza_despesa_codigo,elemento_despesa_codigo
25065,522110100,10111,PO,12243,FS,1500,FR,1001,CO,44905200,...,nan,nan,6700000,ending_balance,D,"6.700.000,00",12243,1500,44905200,00
25066,522110100,10111,PO,12364,FS,1500,FR,1001,CO,33903900,...,nan,nan,6179825,ending_balance,D,"6.179.825,00",12364,1500,33903900,00
25074,522110100,10111,PO,12368,FS,1500,FR,1001,CO,33903200,...,nan,nan,10000,ending_balance,D,"10.000,00",12368,1500,33903200,00
25403,522110100,10111,PO,12368,FS,1500,FR,1001,CO,33903900,...,nan,nan,392254764,ending_balance,D,"392.254.764,00",12368,1500,33903900,00
25407,522110100,10111,PO,12302,FS,1500,FR,1001,CO,44905100,...,nan,nan,1107418,ending_balance,D,"1.107.418,00",12302,1500,44905100,00


In [26]:
# Leia aqui a consulta detalhada extraida do sistema com as colunas da regra.
# Se o nome do arquivo for outro, altere apenas a variavel arquivo_sistema.
df_sistema = pd.read_csv(arquivo_sistema, sep=';', dtype=str, encoding='latin-1', quotechar='"')
df_sistema.columns = [col.strip() for col in df_sistema.columns]

aliases = {
    'fonte_rj': ['FONTE RJ', '[FONTE RJ].[CODIGO]', 'FONTE RJ CODIGO', 'FONTE_RJ', 'CODIGO FONTE RJ', 'FONTE'],
    'funcao_codigo': ['FUNCAO', '[FUNCAO].[CODIGO]', 'FUNCAO CODIGO', 'CODIGO FUNCAO'],
    'sub_funcao_codigo': ['SUB FUNCAO', 'SUB_FUNCAO', '[SUB FUNCAO].[CODIGO]', 'SUB FUNCAO CODIGO', 'CODIGO SUB FUNCAO'],
    'ano_fonte_codigo': ['ANO FONTE', 'ANO_FONTE', '[ANO FONTE].[CODIGO]', 'ANO FONTE CODIGO', 'CODIGO ANO FONTE'],
    'unidade_orcamentaria_codigo': ['UNIDADE_ORCAMENTARIA', 'UNIDADE ORCAMENTARIA', '[UNIDADE ORCAMENTARIA].[CODIGO]', 'UNIDADE ORCAMENTARIA CODIGO', 'CODIGO UNIDADE ORCAMENTARIA'],
    'unidade_gestora_saldo_codigo': ['UNIDADE_GESTORA_DO_SALDO', 'UNIDADE GESTORA DO SALDO', '[UNIDADE GESTORA DO SALDO].[CODIGO]', 'UNIDADE GESTORA DO SALDO CODIGO', 'CODIGO UNIDADE GESTORA DO SALDO'],
    'natureza_despesa_codigo': ['NATUREZA_DESPESA_8_DIGITOS', 'NATUREZA DA DESPESA 8 DIGITOS', '[NATUREZA DA DESPESA 8 DIGITOS].[CODIGO]', 'NATUREZA DA DESPESA CODIGO', 'CODIGO NATUREZA DA DESPESA 8 DIGITOS'],
    'acao_codigo': ['ACAO', '[ACAO].[CODIGO]', 'ACAO CODIGO', 'CODIGO ACAO'],
    'valor': ['VALOR', 'VALOR LIQUIDO', 'VALOR TOTAL'],
}

df_sistema = localizar_colunas(df_sistema, aliases)

for col in df_sistema.columns:
    df_sistema[col] = df_sistema[col].astype(str).str.strip()

df_sistema['valor_num'] = converter_valor(df_sistema['valor'])
df_sistema['funcao_codigo'] = df_sistema['funcao_codigo'].str.zfill(2)
df_sistema['sub_funcao_codigo'] = df_sistema['sub_funcao_codigo'].str.zfill(3)
df_sistema['ano_fonte_codigo'] = df_sistema['ano_fonte_codigo'].str.zfill(1)
df_sistema['fonte_rj'] = df_sistema['fonte_rj'].str.zfill(3)
df_sistema['ic2_equivalente'] = df_sistema['funcao_codigo'] + df_sistema['sub_funcao_codigo']
df_sistema['ic3_equivalente'] = df_sistema['ano_fonte_codigo'] + df_sistema['fonte_rj']
df_sistema['natureza_despesa_codigo'] = df_sistema['natureza_despesa_codigo'].str.zfill(8)
df_sistema['elemento_despesa_codigo'] = df_sistema['natureza_despesa_codigo'].str[2:4]

fontes_validas = ('100', '102', '107', '108', '122', '215')
uos_bloqueadas = ('1241', '2041', '2141', '40410')
nds_bloqueadas = (
    '319001', '319003', '319005', '31900703', '31901308', '31901312',
    '3370', '33900803', '339008', '33903992', '33904723', '339059',
    '33909302', '44909302'
)
acoes_bloqueadas = {'2253', '2701', '8302'}

mask_fonte = df_sistema['fonte_rj'].str.endswith(fontes_validas, na=False)
mask_funcao = df_sistema['funcao_codigo'].str[:2].eq('12')

mask_excecao = (
    df_sistema['unidade_orcamentaria_codigo'].str.startswith(uos_bloqueadas, na=False)
    | df_sistema['unidade_gestora_saldo_codigo'].str.startswith('1234', na=False)
    | df_sistema['natureza_despesa_codigo'].str.startswith(nds_bloqueadas, na=False)
    | df_sistema['elemento_despesa_codigo'].eq('92')
    | df_sistema['acao_codigo'].isin(acoes_bloqueadas)
)

sistema_co1001 = df_sistema.loc[mask_fonte & mask_funcao & ~mask_excecao].copy()

print('Linhas do sistema enquadradas na regra do CO 1001:', len(sistema_co1001))
print('Total do sistema para CO 1001:', sistema_co1001['valor_num'].sum())

sistema_co1001.head()


KeyError: "Colunas nao encontradas no extrato do sistema: ['unidade_orcamentaria_codigo', 'unidade_gestora_saldo_codigo', 'natureza_despesa_codigo', 'elemento_despesa_codigo']"

In [ ]:
# Comparacao resumida por chaves comuns.
# Se quiser uma conferencia mais detalhada, inclua aqui outras chaves que existirem nas duas bases.
chaves = ['ic2_equivalente', 'ic3_equivalente', 'natureza_despesa_codigo']

resumo_matriz = (
    matriz_co1001.groupby(chaves, dropna=False, as_index=False)['VALOR_NUM']
    .sum()
    .rename(columns={'VALOR_NUM': 'valor_matriz'})
)

resumo_sistema = (
    sistema_co1001.groupby(chaves, dropna=False, as_index=False)['valor_num']
    .sum()
    .rename(columns={'valor_num': 'valor_sistema'})
)

comparacao = resumo_matriz.merge(resumo_sistema, on=chaves, how='outer')
comparacao['valor_matriz'] = comparacao['valor_matriz'].fillna(0)
comparacao['valor_sistema'] = comparacao['valor_sistema'].fillna(0)
comparacao['diferenca'] = comparacao['valor_matriz'] - comparacao['valor_sistema']
comparacao['confere'] = np.isclose(comparacao['diferenca'], 0, atol=0.01)

print('Total matriz  :', resumo_matriz['valor_matriz'].sum())
print('Total sistema :', resumo_sistema['valor_sistema'].sum())
print('Dif. total    :', resumo_matriz['valor_matriz'].sum() - resumo_sistema['valor_sistema'].sum())

comparacao.sort_values(['confere', 'diferenca'], ascending=[True, False]).head(50)
